Axl Waltz Cruz<br>
CPE22S3-CPE311<br>
Engr.Neal Matira

In [3]:
#exercise 4
import pandas as pd
import numpy as np


df = pd.read_csv("meteorite-landings.csv")


print(df.columns)


df["year"] = pd.to_datetime(df["year"], errors="coerce").dt.year


df["mass"] = pd.to_numeric(df["mass"], errors="coerce")


df_filtered = df[(df["year"] >= 2005) & (df["year"] <= 2009)]


pivot = df_filtered.pivot_table(
    index="year",
    columns="fall",
    values="mass",
    aggfunc=[
        "count",
        lambda x: np.nanpercentile(x, 95)
    ]
)


pivot.columns = ["count" if col[0]=="count" else "p95"
                 for col in pivot.columns]

print(pivot)



Index(['name', 'id', 'nametype', 'recclass', 'mass', 'fall', 'year', 'reclat',
       'reclong', 'GeoLocation'],
      dtype='object')
Empty DataFrame
Columns: []
Index: []


In [4]:
summary_stats = df.groupby("fall")["mass"].agg([
    "count",
    "mean",
    "median",
    "std",
    "min",
    "max",
    lambda x: np.nanpercentile(x, 95)
])

summary_stats.rename(columns={"<lambda_0>": "p95"}, inplace=True)

print(summary_stats)


       count          mean  median            std  min         max       p95
fall                                                                        
Fell    1075  47070.715023  2800.0  717067.125826  0.1  23000000.0  100000.0
Found  44510  12461.922983    30.5  571105.752311  0.0  60000000.0    2850.0


In [5]:
summary_stats = df.groupby("fall")["mass"].describe()
print(summary_stats)


         count          mean            std  min     25%     50%      75%  \
fall                                                                        
Fell    1075.0  47070.715023  717067.125826  0.1  686.00  2800.0  10450.0   
Found  44510.0  12461.922983  571105.752311  0.0    6.94    30.5    178.0   

              max  
fall               
Fell   23000000.0  
Found  60000000.0  


In [9]:
#exercise 5

!wget -O yellow_tripdata_2019-01.parquet \
https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2019-01.parquet

import pandas as pd


df = pd.read_parquet('yellow_tripdata_2019-01.parquet')

df['tpep_dropoff_datetime'] = pd.to_datetime(df['tpep_dropoff_datetime'], errors='coerce')

cols = ['trip_distance', 'fare_amount', 'tolls_amount', 'tip_amount']
df[cols] = df[cols].apply(pd.to_numeric, errors='coerce').fillna(0)


hourly_data = df.set_index('tpep_dropoff_datetime').resample('H').agg({
    'trip_distance': 'sum',
    'fare_amount': 'sum',
    'tolls_amount': 'sum',
    'tip_amount': 'sum'
})


top_5_tip_hours = hourly_data.sort_values(by='tip_amount', ascending=False).head(5)

print("Top 5 Hours with Most Tips:")
print(top_5_tip_hours)



--2026-02-19 15:56:54--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2019-01.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 13.35.33.10, 13.35.33.98, 13.35.33.83, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|13.35.33.10|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 110439634 (105M) [application/x-www-form-urlencoded]
Saving to: ‘yellow_tripdata_2019-01.parquet’

yellow_tripdata_201 100%[===================>] 105.32M  17.2MB/s    in 7.6s    

2026-02-19 15:57:03 (13.9 MB/s) - ‘yellow_tripdata_2019-01.parquet’ saved [110439634/110439634]



/tmp/ipython-input-2640214359.py:18: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  hourly_data = df.set_index('tpep_dropoff_datetime').resample('H').agg({


Top 5 Hours with Most Tips:
                       trip_distance  fare_amount  tolls_amount  tip_amount
tpep_dropoff_datetime                                                      
2019-01-31 19:00:00         51788.76    237655.26       5746.42    40420.90
2019-01-31 18:00:00         51518.40    244824.78       6345.48    40386.11
2019-01-11 18:00:00         51601.90    245275.78       5670.94    39738.01
2019-01-25 18:00:00         52492.45    248175.08       6627.72    39345.27
2019-01-30 19:00:00         50240.34    228497.80       4818.27    38917.37
